# Estimación de motilidad en videos microscópicos de espermatozoides (VISEM)

**Curso:** Deep Learning, EAFIT · **Integrantes:** _(completar)_ · **Última actualización:** _(completar)_

> Estructura alineada con los lineamientos del profesor: problema y objetivos, arquitectura, datos y preprocesamiento, entrenamiento, resultados y conclusiones.
> Antes de trabajar, leer `AGENTS.md` (contexto y reglas del proyecto).

## 0. Configuración y reproducibilidad

Fijamos semillas y registramos versiones y hardware, porque la sustentación exige describir el entorno de cómputo y los resultados deben poder repetirse.

In [ ]:
import os, random, platform
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    print("PyTorch:", torch.__version__, "| CUDA runtime:", torch.version.cuda)
    print("GPU disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
except ImportError:
    print("PyTorch no está instalado todavía.")

print("Python:", platform.python_version(), "| SO:", platform.system())

### Rutas del proyecto

Las carpetas originales quedaron en la raíz (`visem-dataset/` y `EVISAN/`) en lugar de `data/raw/`; las rutas se ajustaron a esa disposición. En Colab, cambiar `ROOT` a la carpeta de Drive. Los datos originales **no se modifican**.

In [ ]:
ROOT = Path(".").resolve()          # raíz del proyecto (el notebook vive en la raíz, junto a AGENTS.md)
VISEM_DIR = ROOT / "visem-dataset"  # originales de VISEM (no se modifican)
EVISAN_DIR = ROOT / "EVISAN"        # originales de EVISAN (no se modifican)
PROCESSED = ROOT / "data" / "processed"
SPLITS = ROOT / "data" / "splits"
MODELS = ROOT / "models"
FIGURES = ROOT / "reports" / "figures"

for p in (PROCESSED, SPLITS, MODELS, FIGURES):
    p.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", ROOT)
print("VISEM:", VISEM_DIR.exists(), "| EVISAN:", EVISAN_DIR.exists())

## 1. Planteamiento del problema y objetivos

**Título (propuesto).** Estimación de la motilidad espermática a partir de videos microscópicos mediante detección, seguimiento y modelos de Deep Learning (VISEM)

**Problema.** El análisis de motilidad de una muestra seminal se hace manualmente sobre video y consume tiempo del personal de laboratorio. Queremos estimar, a partir del video, los porcentajes de motilidad (progresiva, no progresiva, inmóvil) como apoyo al análisis de calidad seminal. **No se diagnostica fertilidad.**

**Objetivo general.** Desarrollar un sistema de Deep Learning que, a partir de videos microscópicos de VISEM, detecte y siga espermatozoides y estime la motilidad de la muestra, como apoyo al análisis de calidad seminal.

**Objetivos específicos** (cada uno con su criterio de verificación):
1. **Detección y tracking.** Entrenar un detector YOLO con EVISAN (división por secuencia) y evaluar su transferencia a VISEM. *Verificación:* mAP@0.5 y mAP@0.5:0.95 en EVISAN no visto, más revisión manual de una muestra de frames de VISEM.
2. **Características de movimiento y estimación de motilidad.** Extraer conteo, velocidad y linealidad de las trayectorias y usarlas para estimar los porcentajes de motilidad. *Verificación:* MAE y RMSE frente a los valores de laboratorio, con validación cruzada agrupada por participante (media ± desviación).
3. **Comparación y análisis crítico.** Comparar el enfoque anterior con una línea base directa (CNN/modelo de video sobre clips) y analizar cuándo y por qué falla cada uno. *Verificación:* tabla comparativa de métricas por participante no visto y análisis de errores.

**Plan B.** Si la transferencia EVISAN→VISEM falla, la línea base directa pasa a ser el resultado principal y el análisis del fallo se documenta como hallazgo.

### Correo de la entrega preliminar (jueves 29 de octubre de 2026)
- **Destinatario:** correo del profesor _(completar)_
- **Asunto:** Proyecto Final – Grupo #_(completar)_ – Objetivos del proyecto
- **Cuerpo:** título, objetivo general y objetivos específicos de arriba (2 a 3, claros y verificables).

### Mapa de entregables
| Entregable | Fecha | Secciones del notebook |
| --- | --- | --- |
| Correo: título y objetivos | 29-oct-2026 | 1 |
| Informe PDF (máx. 3 páginas): objetivos, datos y splits, modelo, resultados con datos no vistos, conclusiones | 11-nov-2026 | 1 a 6 |
| Presentación (15 min + 5 de preguntas): problema, arquitectura, datos, entrenamiento, resultados, conclusiones | 11-nov-2026 | 1 a 6 |

_Pendiente: confirmar con el compañero la redacción final antes de enviar el correo._

## 2. Base de datos y preprocesamiento

### 2.1 VISEM (dataset principal)
Descripción, procedencia, licencia y cita: ver `AGENTS.md`, sección 2. Hechos verificados en los archivos locales:

- 85 videos `.avi` en `visem-dataset/videos/`, uno por participante; 6 CSV y `descriptions.txt`.
- Los CSV usan **`;` como separador y `,` como decimal**, con BOM (`encoding="utf-8-sig"`), y contienen texto como `Not reported` en columnas numéricas.
- `fatty_acids_spermatoza.csv` (sin la "r") es el nombre real del archivo de ácidos grasos de espermatozoides.
- `videos.csv` tiene columnas `ID;video`.
- Variable objetivo (en `semen_analysis_data.csv`): `Progressive motility (%)`, `Non progressive sperm motility (%)`, `Immotile sperm (%)`. Las tres **suman exactamente 100 en los 85 participantes**: es una composición, no tres variables independientes (ver 2.2 y 3).
- El nombre del video termina con el código de quien lo evaluó (SSW 30, JMA 27, HH 18, OW 7, IVS 3) y 14 nombres traen notas de calidad (`drift`, `minor drift`, `lots of debris`).

In [ ]:
# Exploración inicial de las tablas de VISEM (formato: separador ';', decimal ',', BOM)
csv_files = sorted(VISEM_DIR.glob("*.csv"))
print("CSV encontrados:", [f.name for f in csv_files] or "ninguno")

tables = {
    f.stem: pd.read_csv(f, sep=";", decimal=",", encoding="utf-8-sig", na_values=["Not reported"])
    for f in csv_files
}
for name, df in tables.items():
    print(f"\n== {name}: {df.shape[0]} filas x {df.shape[1]} columnas | nulos: {int(df.isna().sum().sum())}")
    display(df.head(3))

**Observaciones de la exploración (calculadas sobre los CSV locales):**

| Variable | Media | Desv. | Mín | Máx |
| --- | --- | --- | --- | --- |
| Progresiva (%) | 42.7 | 20.1 | 0 | 76 |
| No progresiva (%) | 27.7 | 9.5 | 4 | 54 |
| Inmóvil (%) | 29.6 | 17.7 | 10 | 96 |

- Las tres suman 100: conviene predecir una salida que respete eso (p. ej. softmax o predecir dos y derivar la tercera) y reportar MAE/RMSE de las tres.
- Cada participante tiene **un solo evaluador** y las etiquetas tienen variabilidad entre evaluadores: es un techo de calidad que hay que mencionar en limitaciones.
- _Por completar:_ identificar qué columnas de otras tablas tienen nulos, y revisar si el evaluador o los videos con `drift` se asocian con el error.

### 2.2 División en entrenamiento, validación y prueba

**Regla:** dividir **por participante**, nunca por frame o clip. Los fotogramas consecutivos son casi idénticos; si caen en conjuntos distintos, el modelo "ve" la prueba durante el entrenamiento y la métrica se infla.

Con 85 participantes usaremos validación cruzada agrupada por participante y reportaremos media y desviación estándar.

In [ ]:
# Esqueleto: asignación de participantes a folds (columna de ID verificada: "ID", enteros 1..85)
from sklearn.model_selection import GroupKFold

# participant_ids = tables["videos"]["ID"].values
# gkf = GroupKFold(n_splits=5)
# for fold, (tr_idx, te_idx) in enumerate(gkf.split(participant_ids, groups=participant_ids)):
#     ...
# Guardar la asignación en SPLITS / "folds.csv" para que ambos integrantes usen exactamente los mismos splits.

### 2.3 Extracción de clips y frames (una sola vez)

Decodificar 35 GB de video en cada época es inviable. Extraemos clips cortos a disco **una vez** (resolución y fps reducidos si es necesario) y entrenamos desde ahí. Justificar aquí la longitud del clip, el fps y el tamaño de imagen elegidos.

In [ ]:
# Esqueleto de extracción de clips con OpenCV
import cv2

def extract_clips(video_path, out_dir, clip_len=32, stride=50, resize=(320, 240), gray=True):
    """Guarda clips de `clip_len` frames cada `stride` frames como arrays .npy."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    frames, clip_id = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if gray:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frames.append(cv2.resize(frame, resize))
    cap.release()
    for start in range(0, len(frames) - clip_len + 1, stride):
        clip = np.stack(frames[start:start + clip_len])
        np.save(out_dir / f"clip_{clip_id:04d}.npy", clip)
        clip_id += 1
    return clip_id

# Ojo: este esqueleto carga TODOS los frames en memoria (7 min x 50 fps = ~21.000 frames);
# antes de usarlo, leer por ventanas o submuestrear.
# Ejemplo (descomentar cuando exista un video):
# n = extract_clips(VISEM_DIR / "videos" / "1_09.09.02_SSW.avi", PROCESSED / "1")

### 2.4 Normalización y aumento de datos
_Por completar y justificar._ Ideas a evaluar: normalización por intensidad media del video, aumentos que no alteren la dinámica de movimiento (volteos, pequeñas variaciones de brillo/contraste), y **evitar** aumentos temporales que cambien la velocidad aparente, porque la velocidad es justo lo que se mide.

### 2.5 EVISAN (apoyo para el detector)
Hechos verificados en `EVISAN/`:
- 6.000 imágenes JPEG (`images/1.jpg` … `6000.jpg`), 6.000 XML Pascal VOC (`xml_file/`) y 2 videos `.wmv` (`video/test_200x.wmv`, `test_600x.wmv`).
- Todas las imágenes son de **1200×936**. Imágenes 1–3000 a 200X, 3001–6000 a 600X.
- Dos clases en los XML: `black` (82.349 cajas) y `white` (5.113 cajas). El readme no explica qué significan; **no asumir** su significado hasta revisarlo visualmente.
- Cajas por imagen: 16,7 en promedio a 200X y 12,4 a 600X (máximo 45; ninguna imagen sin anotaciones).
- Los XML traen solo clase y caja, **sin identificador de trayectoria**: EVISAN sirve para evaluar detección, pero no trae verdad de terreno de tracking (ver sección 3).

**División:** por secuencia (bloques de 1.000 imágenes → 6 secuencias, 3 a 200X y 3 a 600X). Con tan pocas secuencias, estratificar por aumento (entrenar con ambos y reservar al menos una secuencia de cada uno) y usar los 2 videos como evaluación externa. El readme no indica a qué donante corresponde cada secuencia, así que no podemos garantizar separación por donante.

In [ ]:
# Esqueleto: conversión de anotaciones XML (Pascal VOC) a formato YOLO
import xml.etree.ElementTree as ET

def voc_to_yolo(xml_path, class_map):
    root = ET.parse(xml_path).getroot()
    w = float(root.find("size/width").text)
    h = float(root.find("size/height").text)
    lines = []
    for obj in root.findall("object"):
        cls = obj.find("name").text
        b = obj.find("bndbox")
        xmin, ymin, xmax, ymax = (float(b.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax"))
        xc, yc = (xmin + xmax) / 2 / w, (ymin + ymax) / 2 / h
        bw, bh = (xmax - xmin) / w, (ymax - ymin) / h
        lines.append(f"{class_map[cls]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    return lines

## 3. Arquitectura del modelo

Para cada componente indicar: arquitectura, **referencia bibliográfica**, modificaciones respecto al original y **por qué es adecuada** para el problema (no basta con nombrarla).

| Componente | Arquitectura candidata | Referencia | Justificación / modificaciones |
| --- | --- | --- | --- |
| Detector | YOLO (Ultralytics) | _(completar)_ | _(completar)_ |
| Tracking | _(p. ej. ByteTrack / SORT)_ | _(completar)_ | _(completar)_ |
| Línea base de video | CNN 2D + agregación temporal / modelo de video liviano | _(completar)_ | _(completar)_ |
| Regresor de motilidad | _(MLP / gradient boosting sobre features de movimiento)_ | _(completar)_ | _(completar)_ |

In [ ]:
# Esqueleto de la línea base: CNN 2D por frame + agregación temporal
import torch
import torch.nn as nn

class FrameEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(64, out_dim)

    def forward(self, x):                      # x: (B*T, 1, H, W)
        return self.fc(self.net(x).flatten(1))

class MotilityBaseline(nn.Module):
    """Predice 3 valores (progresiva, no progresiva, inmóvil) a partir de un clip."""
    def __init__(self, emb=128, n_out=3):
        super().__init__()
        self.enc = FrameEncoder(emb)
        self.temporal = nn.GRU(emb, emb, batch_first=True)
        self.head = nn.Linear(emb, n_out)

    def forward(self, clip):                   # clip: (B, T, 1, H, W)
        B, T = clip.shape[:2]
        z = self.enc(clip.flatten(0, 1)).view(B, T, -1)
        _, h = self.temporal(z)
        return self.head(h[-1])

# print(MotilityBaseline())

## 4. Entrenamiento y configuración experimental

Registrar para cada experimento: función de pérdida, optimizador, learning rate, batch size, épocas, scheduler, uso de AMP, recursos (GPU/CPU/RAM) y tiempo de entrenamiento.

| Experimento | Modelo | Pérdida | Optimizador | LR | Batch | Épocas | Hardware | Notas |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| _(completar)_ | | | | | | | | |

In [ ]:
# Esqueleto de bucle de entrenamiento con AMP y registro de curvas
import time

def train_one_epoch(model, loader, loss_fn, optimizer, device, scaler=None):
    model.train()
    total, n = 0.0, 0
    for clips, targets in loader:
        clips, targets = clips.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=scaler is not None):
            loss = loss_fn(model(clips), targets)
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()
        total += loss.item() * clips.size(0); n += clips.size(0)
    return total / n

@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total, n = 0.0, 0
    for clips, targets in loader:
        clips, targets = clips.to(device), targets.to(device)
        total += loss_fn(model(clips), targets).item() * clips.size(0); n += clips.size(0)
    return total / n

history = {"train_loss": [], "val_loss": []}   # llenar por época y guardar en reports/

### Análisis de las curvas
_Presentar la gráfica no basta: interpretar._ Responder aquí: ¿hay sobreajuste (val_loss sube mientras train_loss baja)? ¿subajuste? ¿converge? ¿qué decisión se tomó a partir de esto (early stopping, regularización, más datos, LR)?

## 5. Resultados y evaluación

- Métricas cuantitativas: MAE y RMSE por tipo de motilidad (regresión) y, si se discretiza, accuracy / F1 / matriz de confusión. Detector: mAP@0.5 y mAP@0.5:0.95, precisión y recall.
- Reportar **media ± desviación** sobre los folds por participante.
- **Evaluación con datos externos** (obligatoria): participantes no vistos en entrenamiento y los 2 videos de prueba de EVISAN.
- Análisis de errores: ¿en qué muestras falla? ¿hay relación con la concentración, la calidad del video o el tipo de movimiento?

In [ ]:
# Esqueleto de métricas de regresión
from sklearn.metrics import mean_absolute_error, mean_squared_error

def regression_report(y_true, y_pred, names=("progresiva", "no_progresiva", "inmovil")):
    rows = []
    for i, n in enumerate(names):
        mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = mean_squared_error(y_true[:, i], y_pred[:, i]) ** 0.5
        rows.append({"variable": n, "MAE": mae, "RMSE": rmse})
    return pd.DataFrame(rows)

## 6. Conclusiones

Relacionar cada conclusión con un objetivo específico y respaldarla con un resultado del notebook. Evitar afirmaciones genéricas. Incluir:
- Qué objetivos se cumplieron y con qué evidencia.
- Limitaciones (85 participantes, cambio de dominio EVISAN→VISEM, resolución de video, etiquetas de laboratorio con variabilidad entre evaluadores).
- Posibles mejoras y trabajo futuro.

## 7. Referencias

- Haugen, T. B. et al. (2019). VISEM: A Multimodal Video Dataset of Human Spermatozoa. MMSys'19. https://dl.acm.org/doi/10.1145/3304109.3325814
- Yan, Y. (2020). EVISAN – a dataset for multi-sperm detection and tracking algorithm development. Zenodo. https://doi.org/10.5281/zenodo.4303768
- _(agregar arquitecturas y librerías usadas)_